# Limpieza y Transformación

En este notebook, abordamos los problemas de acalidad de datos identificados durante la fase de análisis exploratorio de datos(EDA)

El objetivo es preparar tablas limpias,consistentes y estructuradas,listas para ser cargadas en una base de datos.

Durante este proceso se realizarán tareas como conversión de tipos de datos, corrección de nombres de columnas, tratamiento de duplicados y manejo de valores nulos.

# 1.Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# 2.Cargando datasets

In [2]:
customers = pd.read_csv('/content/drive/MyDrive/Invers/olist_customers_dataset.csv')
orders = pd.read_csv('/content/drive/MyDrive/Invers/olist_orders_dataset.csv')
order_items = pd.read_csv('/content/drive/MyDrive/Invers/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/drive/MyDrive/Invers/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/content/drive/MyDrive/Invers/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/Invers/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/Invers/olist_sellers_dataset.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/Invers/olist_geolocation_dataset.csv')
categories = pd.read_csv('/content/drive/MyDrive/Invers/product_category_name_translation.csv')

# 3.Copia de Datasets

In [3]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geolocation_clean = geolocation.copy()
categories_clean = categories.copy()

# 4.Convertir fechas a datetime

**Justificación**:

Actualmente,las variables de tiempo de los pedidos estan almacenados como **objects**.Para poder realizar análisis temporales adecuados, como cálculos de duración entre eventos o análisis de tendencias en el tiempo, es necesario convertir estas columnas al tipo **datetime**.

In [4]:
date_columns= [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for column in date_columns:
    orders_clean[column] = pd.to_datetime(orders_clean[column])

In [5]:
orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


# 5.Corregir nombres de columnas

**Justificación**:

Se detectaron algunos errores tipográficos en los nombres de columnas del dataset de productos.

Entonces para mantener consistencia en los nombres y evitar problemas durante el análisis, se corrigieron estos nombres utilizando la función **rename()**


In [6]:
products_clean = products_clean.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

In [7]:
products_clean

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


## Columnas de tiempo

In [8]:
orders_clean = orders_clean.rename(columns={
    "order_approved_at": "order_approved_timestamp",
    "order_delivered_carrier_date": "order_delivered_carrier_timestamp",
    "order_delivered_customer_date": "order_delivered_customer_timestamp",
    "order_estimated_delivery_date": "order_estimated_delivery_timestamp"
})

**Justificación**:

Usamos este estandar de nombres ya que incluyen fecha y hora en los datos de tiempo de estas columnas.

# 6. Duplicados

**Justificación**:

Ya que la tabla *geolocation* tiene más de 261000 filas duplicadas y que como lo analizamos anteriormente un único **zip_code_prefix** tiene multiples coordenadas porque representa un área y no un punto.Esto en un futuro trae problemas al realizar un Join y provocará un desbordamiento de nuestras tablas.

**Solución**: Agruparemos por **zip_code_prefix** y calcularemos el promedio de las coordenadas, conservando la primera ciudad y estado que aparezca.

In [9]:
print(f"Original geolocation shape: {geolocation.shape}")
geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print(f"Clean geolocation shape: {geolocation_clean.shape}")
print(f"Remaining duplicates: {geolocation_clean.duplicated().sum()}")

Original geolocation shape: (1000163, 5)
Clean geolocation shape: (19015, 5)
Remaining duplicates: 0


# 7.Manejo de valores nulos

## Dataset:**reviews**

**Decisión**: Los valores faltantes en las columnas *review_comment_title* y *review_comment_message* se mantendrán como nulos.

**Justificación**:Los valores faltantes se mantendran como nulos ya que no todos los clientes dejan comentarios escritos al realizar una reseña, por lo que estos valores faltantes representan una ausencia real de información y no un error en los datos.La variable clave para el análisis es la puntuación de la reseña,la cual esta completa.


## Dataset:**orders**

**Decisión**: No se imputarán fechas artificiales porque alteraría el flujo real del pedido

**Justificación**:Los valores faltantes se conservaron porque representan situaciones comerciales reales,como pedidos cancelados o no entregados

## Dataset:**products**

**Decisión**: En lugar de eliminar los registros faltantes de *product_category* se remplazaran con "unknow".Los valores numericos se imputaron usando la mediana

**Justificación**:Al eliminar estos registros provocaría que las transacciones de venta asociadas desaparecieran durante las uniones con la tabla **order_items**, lo que resultaria en informes de ingresos inexactos.Entonces,agregar una categoria de "unknow" conserva la integridad de los datos.

Para las variables numéricas, los valores faltantes se imputaron utilizando la mediana, ya que este estadístico es más robusto frente a valores extremos que la media.

In [10]:
products_clean['product_category_name'] = products_clean[
    'product_category_name'
].fillna('unknown')

In [11]:
products_clean['product_name_length'] = products_clean['product_name_length'].fillna(products_clean['product_name_length'].median())
products_clean['product_description_length'] = products_clean['product_description_length'].fillna(products_clean['product_description_length'].median())
products_clean['product_photos_qty'] = products_clean['product_photos_qty'].fillna(products_clean['product_photos_qty'].median())
products_clean['product_weight_g'] = products_clean['product_weight_g'].fillna(products_clean['product_weight_g'].median())
products_clean['product_length_cm'] = products_clean['product_length_cm'].fillna(products_clean['product_length_cm'].median())
products_clean['product_height_cm'] = products_clean['product_height_cm'].fillna(products_clean['product_height_cm'].median())
products_clean['product_width_cm'] = products_clean['product_width_cm'].fillna(products_clean['product_width_cm'].median())

In [12]:
products_clean.isnull().sum()

,0
product_id,0
product_category_name,0
product_name_length,0
product_description_length,0
product_photos_qty,0
product_weight_g,0
product_length_cm,0
product_height_cm,0
product_width_cm,0


# 8.Valores Atípicos

**Justificación**:

Durante el análisis se identificaron valores atípicos en variables relacionadas con precios, costos de envío y valores de pago.Sin embargo, estos valores no representan errores en los datos, sino que corresponden a pedidos reales de alto valor dentro del sistema de comercio electrónico.

Entonces, eliminar estos registros podría provocar una pérdida de información relevante sobre el comportamiento real de las compras.
Por lo tanto, se decidió **mantener los valores atípicos** en el dataset, asegurando que el análisis posterior represente adecuadamente la variabilidad real de los pedidos.

In [13]:
def detectar_outliers(df, columna):

    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]

    print(f"Columna: {columna}")
    print(f"Cantidad de outliers: {len(outliers)}")
    print(f"Porcentaje: {(len(outliers)/len(df))*100:.2f}%")

## Atípicos en variables importantes

In [14]:
detectar_outliers(order_items, 'price')
detectar_outliers(order_items, 'freight_value')
detectar_outliers(payments, 'payment_value')

Columna: price
Cantidad de outliers: 8427
Porcentaje: 7.48%
Columna: freight_value
Cantidad de outliers: 12134
Porcentaje: 10.77%
Columna: payment_value
Cantidad de outliers: 7981
Porcentaje: 7.68%


# 9.Guardar datos limpios

In [15]:
customers_clean.to_csv("customers_clean.csv", index=False)
orders_clean.to_csv("orders_clean.csv", index=False)
order_items_clean.to_csv("order_items_clean.csv", index=False)
payments_clean.to_csv("payments_clean.csv", index=False)
reviews_clean.to_csv("reviews_clean.csv", index=False)
products_clean.to_csv("products_clean.csv", index=False)
sellers_clean.to_csv("sellers_clean.csv", index=False)
geolocation_clean.to_csv("geolocation_clean.csv", index=False)
categories_clean.to_csv("categories_clean.csv", index=False)